# 04 — Gemma 3 4B: textbook-knowledge pilot (one subject)

Does training on the **official textbook** make the model better at that subject's MCQs?
Tested on one subject first: **Christianity, grade 8**, which has the most grade-8 questions in the Dev Set (70).

| Condition | Training data |
|---|---|
| `base` | none (Gemma 3 4B as released) |
| `book` | continued pretraining on the textbook + generated paraphrases |
| `mcq` | MCQs generated from the textbook (answer-only loss) |
| `book+mcq` | `book` first, then the MCQs |

**Evaluation sets** (Dev Set, never used for training):
* **target**: Christianity grade 8 (70). The textbook should help here.
* **control**: Christianity grades 6–7 (60). Different syllabus content, so smaller gains expected.
* **all Dev**: all 1,851 questions, to catch forgetting in other subjects.

With 70 target questions, one question is 1.4 points, and differences below about 10 points can be noise.
The last section runs a paired test against `base`.

**Inputs** (must be in the repo before running on Colab):
* `data/textbooks/christianity_g8_si.jsonl`: from `python -m src.textbooks data/textbooks/christianity_g8_si.pdf`
* `data/synthetic/christianity_g8_{paraphrases,mcq}.jsonl`: from `python -m src.synth ...` (needs a generator API key).
  Without them, only `base` and `book` (textbook text alone) run.

**Hardware:** an NVIDIA GPU. Gemma 3 overflows in float16, so on a **T4** (no bf16) it runs in float32 compute: it works, but slowly
(roughly 2 h end to end). An **L4 / A100** (bf16) is several times faster.

## 1. Environment

In [9]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/Imalsha-Ranasinghe/SinhalaMMLU-Shared-Task.git"
BRANCH   = "dataset/analysis"     # the branch holding src/synth.py and data/synthetic/; "main" once merged

if (Path.cwd() / "src").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    ROOT = Path.cwd().parent
else:
    # Fresh Colab / Kaggle session: fetch the project (the branch must be pushed to GitHub first)
    !git clone -q -b {BRANCH} {REPO_URL}
    ROOT = Path.cwd() / "SinhalaMMLU-Shared-Task"

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

%pip install -q -r requirements-hf.txt

Project root: /content/SinhalaMMLU-Shared-Task


In [10]:
import torch
assert torch.cuda.is_available(), "No GPU. On Colab: Runtime -> Change runtime type -> GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float32     # Gemma 3 overflows in float16
print("compute dtype:", DTYPE)

GPU 0: Tesla T4, 15.6 GB
compute dtype: torch.bfloat16


In [11]:
# Hugging Face login (Gemma 3 is a gated model; accept the licence on its model page first)
from huggingface_hub import login

token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
login(token=token) if token else login()

In [12]:
from google.colab import userdata
from huggingface_hub import whoami, hf_hub_download

tok = userdata.get("HF_TOKEN")
me = whoami(token=tok)
print("token belongs to:", me["name"], "| token type:", me.get("auth", {}).get("accessToken", {}).get("role"))
try:
    hf_hub_download("google/gemma-3-4b-it", "config.json", token=tok)
    print("Gemma access: OK")
except Exception as e:
    print("Gemma access: NO ->", str(e).splitlines()[-1][:150])


token belongs to: Ninada23 | token type: fineGrained
Gemma access: NO -> Access to model google/gemma-3-4b-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-3-4b-it to ask fo


## 2. Configuration — edit this cell

In [13]:
MODEL_ID     = "unsloth/gemma-3-4b-it"     # ungated copy with identical weights: "unsloth/gemma-3-4b-it"
SHORT_NAME   = "gemma3-4b-it"
MAX_PARAMS   = 8e9
LOAD_IN_4BIT = True
SEED         = 42

SUBJECT, GRADE  = "Christianity", 8
BOOK            = ROOT / "data" / "textbooks" / "christianity_g8_si.jsonl"
SYNTH_STEM      = ROOT / "data" / "synthetic" / "christianity_g8"
USE_PARAPHRASES = True
MAX_ANSWER_LENGTH_RATIO = 1.2   # drop MCQs whose answer is >1.2x longer than every wrong option

CONDITIONS    = ["book", "mcq", "book+mcq"]    # base is always evaluated first; book+mcq needs book
EVAL_FULL_DEV = True                           # all 1,851 Dev questions (forgetting check)
SCORER_TOKENS = 8000                           # lower it on CUDA OOM

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
# Text layers only: Gemma 3 4B also contains a vision tower whose layers have the same names.
LORA_TARGETS = r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
CPT_SEQ_LEN, CPT_EPOCHS, CPT_LR = 512, 3, 1e-4
MCQ_EPOCHS, MCQ_LR, MCQ_MAX_LEN = 3, 2e-4, 1024
BATCH_SIZE, GRAD_ACCUM = 2, 8                  # effective batch = 16

RESULTS_DIR = ROOT / "results" / f"{SHORT_NAME}_textbook_pilot"
ADAPTER_DIR = ROOT / "outputs" / f"{SHORT_NAME}_textbook_pilot"

## 3. Parameter budget check

In [14]:
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM

with init_empty_weights():
    _m = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID))
n_params = sum(p.numel() for p in _m.parameters())
del _m
print(f"{MODEL_ID}: {n_params / 1e9:.2f}B parameters incl. vision tower (limit {MAX_PARAMS / 1e9:.0f}B)")
assert n_params < MAX_PARAMS

unsloth/gemma-3-4b-it: 4.97B parameters incl. vision tower (limit 8B)


## 4. Data

In [15]:
import json
import pandas as pd
from src.data import load_raw, validate, load_jsonl

dev, _ = validate(load_raw(ROOT / "Dev Set"))
in_subject = dev["subject"].str.lower() == SUBJECT.lower()
eval_sets = {
    "target": dev[in_subject & (dev["grade"] == GRADE)],
    "control": dev[in_subject & (dev["grade"] != GRADE)],
}
if EVAL_FULL_DEV:
    eval_sets["all_dev"] = dev
for name, d in eval_sets.items():
    print(f"{name:8s} {len(d):5d} questions")

target      70 questions
control     60 questions
all_dev   1851 questions


In [16]:
from src.textbooks import load_pages, passages

book_texts = [p["text"] for p in passages(load_pages(BOOK))]
para_path = Path(f"{SYNTH_STEM}_paraphrases.jsonl")
mcq_path = Path(f"{SYNTH_STEM}_mcq.jsonl")
para_texts = load_jsonl(para_path)["text"].tolist() if USE_PARAPHRASES and para_path.exists() else []
mcq_df = load_jsonl(mcq_path) if mcq_path.exists() else None

print(f"textbook passages: {len(book_texts)}   paraphrases: {len(para_texts)}   "
      f"generated MCQs: {0 if mcq_df is None else len(mcq_df)}")
report_path = Path(f"{SYNTH_STEM}_report.json")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
if mcq_df is None:
    CONDITIONS = [c for c in CONDITIONS if "mcq" not in c]
    print("No generated MCQs found: running", ["base"] + CONDITIONS)
else:
    # The generator never saw Dev questions, and src.synth already dropped near matches. Exact-match check:
    overlap = set(mcq_df["question"].str.strip()) & set(dev["question"].str.strip())
    assert not overlap, f"{len(overlap)} generated questions are identical to Dev questions"

    # Length shortcut: if the correct option is usually the longest, the model learns "pick the longest"
    # instead of the subject. src.synth filters this already; enforce it again here in case the file
    # came from an older run or another generator.
    from src.synth import answer_length_ratio

    def longest_is_answer(d):
        return d.apply(lambda r: max(range(4), key=lambda i: len(r["choices"][i])) + 1 == r["answer"], axis=1).mean()

    n_before, rate_before = len(mcq_df), longest_is_answer(mcq_df)
    mcq_df = mcq_df[mcq_df.apply(answer_length_ratio, axis=1) <= MAX_ANSWER_LENGTH_RATIO].reset_index(drop=True)
    rate_after = longest_is_answer(mcq_df)
    print(f"length check: {n_before} -> {len(mcq_df)} MCQs; correct option is the longest in "
          f"{rate_before:.0%} -> {rate_after:.0%} of them (Dev Set: ~30%, no bias: 25%)")
    assert rate_after <= 0.35, "MCQs still favour the longest option: lower MAX_ANSWER_LENGTH_RATIO or regenerate"
    print(mcq_df["answer"].value_counts().sort_index().rename("answer position").to_frame().T)

textbook passages: 122   paraphrases: 244   generated MCQs: 349
{
  "date": "2026-10-06",
  "generator_model": "gemma-4-31b-it",
  "source": "data/textbooks/christianity_g8_si.jsonl",
  "passages": 122,
  "paraphrases": 244,
  "mcq_parsed_valid": 546,
  "mcq_rejected": {
    "evidence not in passage": 20,
    "all/none of the above": 2,
    "not 4 options": 3
  },
  "mcq_dropped": {
    "near_duplicate": 3,
    "too_close_to_dev": 6,
    "answer_much_longer_than_others": 188
  },
  "mcq_final": 349
}
length check: 349 -> 349 MCQs; correct option is the longest in 32% -> 32% of them (Dev Set: ~30%, no bias: 25%)
answer            1   2   3   4
answer position  88  87  87  87


## 5. Helpers

In [17]:
import gc, math, shutil
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from src.hf_model import (load_model_and_tokenizer, ChoiceScorer, build_lm_examples,
                          build_train_examples, Collator, make_trainer_class, render_prompt)
from src.prompting import build_messages
from src.evaluate import run_eval

def load_base():
    return load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT, dtype=DTYPE)

def prepare_for_training(model):
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    return model

def add_lora(model):
    return get_peft_model(prepare_for_training(model), LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, task_type="CAUSAL_LM"))

def train(model, tok, examples, epochs, lr):
    steps = math.ceil(len(examples) / (BATCH_SIZE * GRAD_ACCUM)) * epochs
    args = TrainingArguments(
        output_dir=str(ROOT / "outputs" / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=epochs, learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * steps)), logging_steps=5, save_strategy="no",
        bf16=BF16, fp16=False,                      # never fp16 with Gemma 3
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False, report_to="none", seed=SEED,
    )
    trainer = make_trainer_class()(model=model, args=args, train_dataset=examples,
                                   data_collator=Collator(tok.pad_token_id))
    trainer.train()
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    if losses:
        print(f"training loss: first {losses[0]:.3f} -> last {losses[-1]:.3f}")

def evaluate(model, tok, name):
    # Fresh predictions every time (run_eval would otherwise resume from an older run's file).
    shutil.rmtree(RESULTS_DIR / name, ignore_errors=True)
    model.config.use_cache = True
    scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
    out = {}
    for set_name, d in eval_sets.items():
        preds = run_eval(d, scorer, lambda r: [], RESULTS_DIR / name / f"{set_name}.jsonl", batch_size=64)
        out[set_name] = preds["correct"].mean()
        print(f"  {name:9s} {set_name:8s} accuracy = {out[set_name]:.1%}")
    return out

## 6. Base model

In [18]:
model, tok = load_base()
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(render_prompt(tok, build_messages(eval_sets["target"].iloc[0].to_dict())))

# Gemma 3 + float16 produces inf/NaN; make sure the logits are finite before spending GPU time.
probe = ChoiceScorer(model, tok)
enc = tok(render_prompt(tok, build_messages(eval_sets["target"].iloc[0].to_dict())),
          add_special_tokens=False, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**enc).logits[0, -1, probe.option_ids]
print("option logits:", logits.tolist())
assert torch.isfinite(logits).all(), "non-finite logits: use bf16 or float32, not float16"
del probe, enc

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

GPU memory used: 3.2 GB
<bos><start_of_turn>user
You are an expert at answering multiple-choice questions written in Sinhala. Read the question and the numbered options, then reply with ONLY the number of the correct option (for example: 2). Do not add any explanation.

Question: දෙවියන් වහන්සේගේ ස්වරූපය හා සමානාත්මතාවය අනුව මිනිසා මවන ලද බව සඳහන් වන ග්‍රන්ථය වන්නේ ,

1. එළිදරව්ව ය
2. උත්පත්තිය
3. නික්මයාම ය
4. ලුක් ය

Answer:<end_of_turn>
<start_of_turn>model

option logits: [26.5, 24.25, 18.625, 17.375]


In [19]:
summary = {"base": evaluate(model, tok, "base")}

book_examples = build_lm_examples(tok, book_texts + para_texts, CPT_SEQ_LEN)
mcq_examples = build_train_examples(tok, mcq_df, max_len=MCQ_MAX_LEN) if mcq_df is not None else []
print(f"continued-pretraining sequences: {len(book_examples)} x {CPT_SEQ_LEN} tokens   MCQ examples: {len(mcq_examples)}")

0 already done, 70 to go -> /content/SinhalaMMLU-Shared-Task/results/gemma3-4b-it_textbook_pilot/base/target.jsonl


  0%|          | 0/70 [00:00<?, ?it/s]

  base      target   accuracy = 25.7%
0 already done, 60 to go -> /content/SinhalaMMLU-Shared-Task/results/gemma3-4b-it_textbook_pilot/base/control.jsonl


  0%|          | 0/60 [00:00<?, ?it/s]

  base      control  accuracy = 38.3%
0 already done, 1851 to go -> /content/SinhalaMMLU-Shared-Task/results/gemma3-4b-it_textbook_pilot/base/all_dev.jsonl


  0%|          | 0/1851 [00:00<?, ?it/s]

  base      all_dev  accuracy = 37.1%
continued-pretraining sequences: 473 x 512 tokens   MCQ examples: 349


## 7. Train and evaluate each condition

Every condition starts from a freshly loaded base model. `book+mcq` starts from the saved `book` adapter.
Epochs are fixed in advance: no Dev questions are used to choose checkpoints, so the Dev scores stay honest.

In [ ]:
assert "book+mcq" not in CONDITIONS or CONDITIONS.index("book") < CONDITIONS.index("book+mcq")

for cond in CONDITIONS:
    print(f"\n===== {cond} =====")
    del model                      # free the previous model before loading the next one
    gc.collect(); torch.cuda.empty_cache()
    model, tok = load_base()
    torch.manual_seed(SEED)
    if cond == "book+mcq":
        model = PeftModel.from_pretrained(prepare_for_training(model), str(ADAPTER_DIR / "book"), is_trainable=True)
    else:
        model = add_lora(model)
    model.print_trainable_parameters()

    if cond == "book":
        train(model, tok, book_examples, CPT_EPOCHS, CPT_LR)
    if cond in ("mcq", "book+mcq"):
        train(model, tok, mcq_examples, MCQ_EPOCHS, MCQ_LR)

    model.save_pretrained(str(ADAPTER_DIR / cond))
    summary[cond] = evaluate(model, tok, cond)


===== book =====


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

trainable params: 29,802,496 || all params: 4,329,881,968 || trainable%: 0.6883


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,27.759680
10,19.150854
15,17.207176


...

  0%|          | 0/70 [00:00<?, ?it/s]

  book      target   accuracy = 42.9%
0 already done, 60 to go -> /content/SinhalaMMLU-Shared-Task/results/gemma3-4b-it_textbook_pilot/book/control.jsonl


  0%|          | 0/60 [00:00<?, ?it/s]

  book      control  accuracy = 38.3%
0 already done, 1851 to go -> /content/SinhalaMMLU-Shared-Task/results/gemma3-4b-it_textbook_pilot/book/all_dev.jsonl


  0%|          | 0/1851 [00:00<?, ?it/s]

## 8. Results

In [1]:
res = pd.DataFrame(summary).T
res.to_csv(RESULTS_DIR / "summary.csv")
n = {k: len(v) for k, v in eval_sets.items()}
display(res.style.format("{:.1%}").set_caption(f"Accuracy  (n = {n})"))

# Change vs base on the target set, with a paired (McNemar exact) test on the questions that flipped.
from scipy.stats import binomtest

def correct_by_id(name, set_name="target"):
    return pd.read_json(RESULTS_DIR / name / f"{set_name}.jsonl", lines=True).set_index("id")["correct"]

base_t = correct_by_id("base")
rows = []
for cond in CONDITIONS:
    c = correct_by_id(cond).reindex(base_t.index)
    gained, lost = int((c & ~base_t).sum()), int((~c & base_t).sum())
    p = binomtest(gained, gained + lost).pvalue if gained + lost else 1.0
    rows.append({"condition": cond, "target Δ vs base": c.mean() - base_t.mean(),
                 "questions gained": gained, "questions lost": lost, "p (McNemar)": p})
pd.DataFrame(rows).set_index("condition").style.format(
    {"target Δ vs base": "{:+.1%}", "p (McNemar)": "{:.3f}"})

NameError: name 'pd' is not defined

In [1]:
# Target accuracy by exam year: 2019 papers vs 2022-23 papers (in case the syllabus changed).
year = eval_sets["target"].set_index("id")["year"]
by_year = {name: correct_by_id(name).groupby(year).mean() for name in summary}
pd.DataFrame(by_year).T.style.format("{:.1%}").set_caption(
    "Target accuracy by exam year  (n = " + str(year.value_counts().sort_index().to_dict()) + ")")

NameError: name 'eval_sets' is not defined

### Reading the results

* **book > base on target, not on control** → the model learned subject knowledge from the book.
  Scale up: more grade-8 books, then grades 6–7 and 9–11.
* **mcq ≥ book+mcq > book** → most of the gain is the answer format, not the knowledge. Generated MCQs matter more.
* **all_dev drops** for any condition → forgetting. Mix some general Sinhala MCQs into training, or lower the epochs or LR.
* **Nothing is significant** → 70 questions is too few to tell. Add more subjects before drawing conclusions.

Adapters are in `outputs/gemma3-4b-it_textbook_pilot/<condition>/`, predictions in `results/gemma3-4b-it_textbook_pilot/`.
Colab deletes both when the session ends; copy them to Drive if you want to keep them.